In [ ]:
from google.colab import files

# This will open a file chooser to select your CSV
uploaded = files.upload()


Saving dataset - Sheet1.csv to dataset - Sheet1 (2).csv


In [ ]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('dataset - Sheet1 (2).csv')

# Check the first few rows to make sure it loaded correctly
df.head()

,text,label
0,That was slick bro 🔥,Positive
1,Bro that move was clean fr,Positive
2,Mess food actually slaps today 😳,Positive
3,Got a 9 GPA no cap 🥶,Positive
4,This fit is bussin’ 🔥🔥,Positive


In [ ]:
# Function to clean text
def clean_text(text):
    text = str(text).lower()   # lowercase
    text = text.strip()         # remove leading/trailing spaces
    return text

# Apply to the text column
df['text'] = df['text'].apply(clean_text)

# Check the first 5 rows after cleaning
df.head()


,text,label
0,that was slick bro 🔥,Positive
1,bro that move was clean fr,Positive
2,mess food actually slaps today 😳,Positive
3,got a 9 gpa no cap 🥶,Positive
4,this fit is bussin’ 🔥🔥,Positive


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Step 1: Split into train (80%) and temp (20% → will be split into val/test)
X_train, X_temp, y_train, y_temp = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

# Step 2: Split temp into validation (10%) and test (10%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Check sizes
print("Train samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

# Step 3: Vectorize text using CountVectorizer (includes emojis automatically)
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print("Vectorization done! Vocabulary size:", len(vectorizer.vocabulary_))


Train samples: 987
Validation samples: 123
Test samples: 124
Vectorization done! Vocabulary size: 1249


In [ ]:
# Temporary emoji dictionary for testing
emoji_dict = {
  # Positive / Chill / Fun
    "🔥": "emoji_fire",
    "😎": "emoji_cool",
    "😂": "emoji_laugh",
    "❤️": "emoji_love",
    "😳": "emoji_surprised",
    "🥶": "emoji_cold",
    "🙌": "emoji_hands_up",
    "💪": "emoji_flex",
    "🏆": "emoji_trophy",
    "🤩": "emoji_excited",
    "👑": "emoji_crown",
    "✨": "emoji_sparkle",
    "🎧": "emoji_headphones",
    "🍕": "emoji_pizza",
    "🎂": "emoji_cake",
    "🎮": "emoji_game",

    # Negative / Frustration
    "💀": "emoji_skull",
    "😬": "emoji_sad",
    "😡": "emoji_angry",
    "😤": "emoji_frustrated",
    "😢": "emoji_crying",
    "😭": "emoji_crying_hard",
    "💔": "emoji_broken_heart",
    "😖": "emoji_annoyed",
    "😩": "emoji_tired",
    "🤯": "emoji_mind_blown",
    "😱": "emoji_shocked",
    "👿": "emoji_angry_face",

    # Humorous / Playful
    "🤣": "emoji_lol",
    "😹": "emoji_laugh_cat",
    "😜": "emoji_tongue",
    "🙃": "emoji_upside_down",
    "😏": "emoji_smirk",
    "🤪": "emoji_silly",
    "🫣": "emoji_peeking"
}

def replace_emojis(text):
    for emo, desc in emoji_dict.items():
        text = text.replace(emo, f" {desc} ")
    return text

# Apply to all splits
X_train = X_train.apply(replace_emojis)
X_val = X_val.apply(replace_emojis)
X_test = X_test.apply(replace_emojis)


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

max_len = 20  # adjust based on your sentences
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# Convert to NumPy arrays to avoid retracing warning
X_train_pad = np.array(X_train_pad)
X_val_pad = np.array(X_val_pad)
X_test_pad = np.array(X_test_pad)


In [ ]:
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder

# Encode text labels to integers
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

# Convert integers to one-hot vectors
y_train_cat = to_categorical(y_train_enc)
y_val_cat = to_categorical(y_val_enc)
y_test_cat = to_categorical(y_test_enc)

# Check number of classes
num_classes = y_train_cat.shape[1]
print("Number of sentiment classes:", num_classes)




Number of sentiment classes: 5


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 128  # increased from 50
max_len = X_train_pad.shape[1]

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim))
model.add(Bidirectional(LSTM(128)))  # Bidirectional LSTM
model.add(Dropout(0.3))
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_6 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=15,       # increase if needed
    batch_size=32,
    verbose=1
)


Epoch 1/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.3025 - loss: 1.5628 - val_accuracy: 0.6016 - val_loss: 1.1400
Epoch 2/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6686 - loss: 0.8939 - val_accuracy: 0.7886 - val_loss: 0.5759
Epoch 3/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8345 - loss: 0.4266 - val_accuracy: 0.8537 - val_loss: 0.3873
Epoch 4/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9418 - loss: 0.1781 - val_accuracy: 0.8537 - val_loss: 0.4419
Epoch 5/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9665 - loss: 0.0960 - val_accuracy: 0.8862 - val_loss: 0.3395
Epoch 6/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9891 - loss: 0.0483 - val_accuracy: 0.8780 - val_loss: 0.4707
Epoch 7/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9955 - loss: 0.0273 - val_accuracy: 0.8699 - val_loss: 0.4717
Epoch 8/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9963 - loss: 0.0214 - val_accuracy: 0.8780 - val_l

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Convert to NumPy arrays
X_test_pad = np.array(X_test_pad)
y_test_cat = np.array(y_test_cat)

# Predict classes
y_pred_probs = model.predict(X_test_pad, batch_size=32, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

# Classification report
report = classification_report(y_true, y_pred, target_names=label_encoder.classes_)
print("Classification Report:\n", report)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)


Classification Report:
               precision    recall  f1-score   support

 Frustration       1.00      1.00      1.00        20
    Humorous       1.00      1.00      1.00        30
    Negative       0.83      0.87      0.85        23
     Neutral       0.85      0.97      0.91        30
    Positive       0.94      0.71      0.81        21

    accuracy                           0.92       124
   macro avg       0.92      0.91      0.91       124
weighted avg       0.92      0.92      0.92       124

Confusion Matrix:
 [[20  0  0  0  0]
 [ 0 30  0  0  0]
 [ 0  0 20  2  1]
 [ 0  0  1 29  0]
 [ 0  0  3  3 15]]


In [ ]:
import numpy as np

np.save("X_train_pad.npy", X_train_pad)
np.save("y_train_cat.npy", y_train_cat)
np.save("X_test_pad.npy", X_test_pad)
np.save("y_test_cat.npy", y_test_cat)
